# 02 — Feature quality and leakage checks

The feature set must be computable at date `t` using information dated no later than `t`. This notebook audits missingness, distributions, cross-asset features and basic leakage indicators.

In [1]:
from pathlib import Path
import os
import sys
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'src').exists(): ROOT = (ROOT / '..').resolve()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / 'src'))
from gold_silver.config import load_config
from gold_silver.data import load_cached_market_data
from gold_silver.features import build_features

config = load_config(ROOT / 'configs/default.yaml')
features = build_features(load_cached_market_data(config), config.features)
missing = features.isna().mean().sort_values(ascending=False)
print(f'{features.shape[0]:,} rows × {features.shape[1]} columns')
display(missing.head(20).to_frame('missing_fraction'))

6,510 rows × 148 columns


,missing_fraction
silver_volume_change,0.113671
gold_silver_ratio_change,0.105837
gold_volume_change,0.064363
silver_return_lag_60,0.009370
gold_return_lag_60,0.009370
silver_volatility_60,0.009217
silver_return_mean_60,0.009217
gold_silver_return_corr_60,0.009217
silver_momentum_60,0.009217
gold_volatility_60,0.009217


In [2]:
sns.set_theme(style='whitegrid', context='notebook')
fig, axes = plt.subplots(1, 2, figsize=(15, 5), constrained_layout=True)
missing.head(20).sort_values().plot(kind='barh', ax=axes[0], color='#c47f00', title='Top feature missingness')
axes[0].set_xlabel('Fraction missing')
features[['gold_return_current', 'silver_return_current', 'gold_volatility_20', 'silver_volatility_20']].rolling(20).mean().plot(ax=axes[1], title='Smoothed feature examples')
axes[1].set_ylabel('Value')
plt.show()

/var/folders/ys/hvb2c_0n7lb1bsh31xyf6d2h0000gp/T/ipykernel_87549/3280308427.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [3]:
selected = [c for c in features.columns if any(k in c for k in ['gold_return', 'silver_return', 'dxy_return', 'vix_return'])]
corr = features[selected].corr()
plt.figure(figsize=(12, 9))
sns.heatmap(corr, cmap='vlag', center=0, xticklabels=False, yticklabels=True)
plt.title('Feature correlation heatmap')
plt.show()

forbidden = [c for c in features.columns if 'target' in c.lower() or 'future' in c.lower()]
assert not forbidden, f'Potential future-looking columns found: {forbidden}'
print('Leakage name check passed: no target/future feature columns.')
print('Rows after dropping the initialization history:', len(features.dropna()))

Leakage name check passed: no target/future feature columns.
Rows after dropping the initialization history: 4850


/var/folders/ys/hvb2c_0n7lb1bsh31xyf6d2h0000gp/T/ipykernel_87549/3429721458.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**How to read the plots.** Missingness is the fraction of dates without a value; it must be explained rather than silently filled with future information. The heatmap shows redundancy among predictors, and the leakage check verifies that feature names and dates do not reveal the target.